# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashkrverma1234-glitch/ml-internship-assignment1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The shipped scorer is the **leakage-clean logistic regression from ML-09** (no
`impressions_90d`/`clicks_90d`/`sessions_90d`), refit on all 32 clients — not the inflated
Week-5 model. Final score = `0.65 x model_probability + 0.35 x normalized Week-4 baseline
score`. Reason codes: `model_decline_risk` (probability >= 0.60), `ctr_gap_vs_position_tier`
(Week-4's rule fired), `model_and_rule_agree` when both do. Three actions: `refresh_priority`
(both signals agree), `refresh_review` (one signal), `monitor` (neither) — see the top-10 queue
below. Note several top rows show `trend_direction` of "up" or "stable", not "down": this queue
flags pages **worth reviewing for opportunity**, not only pages already confirmed declining —
that distinction matters for how a content team should read it.


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashkrverma1234-glitch/ml-internship-assignment1"
REPO_DIR = "ml-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

os.makedirs("work/outputs", exist_ok=True)
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Week-4 baseline score, unchanged.
tier_median_ctr = df.loc[df["avg_position"] > 0].groupby("position_tier")["ctr"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)
VOLUME_FLOOR = 500
eligible = (df["avg_position"] > 0) & (df["impressions_90d"] >= VOLUME_FLOOR) & df["tier_median_ctr"].notna()
stale_bonus = np.where(df["days_since_last_update"] >= 180, 1.15, 1.0)
df["ctr_gap"] = np.where(eligible, (df["tier_median_ctr"] - df["ctr"]).clip(lower=0), 0.0)
df["baseline_action_score"] = np.where(eligible, df["ctr_gap"] * np.log1p(df["impressions_90d"]) * stale_bonus, 0.0)

# LEAKAGE-CLEAN feature set: after ML-09's audit found impressions_90d/clicks_90d/sessions_90d
# overlap the label's own 30-day windows, this playbook ships the model WITHOUT them --
# a real but smaller lift is worth more than an inflated one for something going in the paper.
CLEAN_NUMERIC_FEATURES = ["ctr", "avg_position", "content_age_days", "days_since_last_update", "word_count"]
CATEGORICAL_FEATURES = [c for c in ["content_type", "position_tier", "main_intent"] if c in df.columns]
num = df[CLEAN_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
cat = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
cat_dummies = pd.get_dummies(cat, prefix=CATEGORICAL_FEATURES, dtype=float)
X_clean = pd.concat([num.reset_index(drop=True), cat_dummies.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

# Production refit: trained on ALL clients (not held out), since this queue ships to every
# client, not just the 26 used for training in Week-5/9's evaluation split. Expected
# performance is NOT re-measured here (that would be in-sample and dishonest) -- it's carried
# over from ML-09's held-out, leakage-clean numbers: precision@50 ~0.46, ROC AUC ~0.707.
model = Pipeline([("scaler", StandardScaler()),
                   ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
model.fit(X_clean, y)
df["model_probability"] = model.predict_proba(X_clean)[:, 1]

# Blend: mostly the model, with the transparent rule as a minority vote -- if they agree, trust
# the score more; if they disagree, the reason codes below say so explicitly.
def normalize(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0

df["baseline_score_normalized"] = normalize(df["baseline_action_score"])
df["final_action_score"] = 100 * (0.65 * df["model_probability"] + 0.35 * df["baseline_score_normalized"])

def reason_codes(row):
    reasons = []
    if row["model_probability"] >= 0.60:
        reasons.append("model_decline_risk")
    if row["baseline_action_score"] > 0:
        reasons.append("ctr_gap_vs_position_tier")
    if row["model_probability"] >= 0.60 and row["baseline_action_score"] > 0:
        reasons.append("model_and_rule_agree")
    if not reasons:
        reasons.append("general_review")
    return "|".join(reasons)

def suggested_action(reasons):
    r = set(reasons.split("|"))
    if "model_and_rule_agree" in r:
        return "refresh_priority"
    if "model_decline_risk" in r or "ctr_gap_vs_position_tier" in r:
        return "refresh_review"
    return "monitor"

df["reason_codes"] = df.apply(reason_codes, axis=1)
df["suggested_action"] = df["reason_codes"].apply(suggested_action)
df["rank"] = df["final_action_score"].rank(method="first", ascending=False).astype(int)

queue = df.sort_values("rank")
cols = ["rank", "content_id", "client_id", "final_action_score", "model_probability",
        "baseline_action_score", "reason_codes", "suggested_action", "impressions_90d",
        "avg_position", "days_since_last_update", "trend_direction"]
print(f"Queue built for all {len(queue):,} pages across {queue['client_id'].nunique()} clients.")
print(f"Action mix: {dict(queue['suggested_action'].value_counts())}")
print()
print("Top 10:")
print(queue[cols].head(10).round(3).to_string(index=False))


Queue built for all 30,000 pages across 32 clients.
Action mix: {'monitor': 16924, 'refresh_review': 11355, 'refresh_priority': 1721}

Top 10:
 rank           content_id         client_id  final_action_score  model_probability  baseline_action_score                                                     reason_codes suggested_action  impressions_90d  avg_position  days_since_last_update trend_direction
    1 content_453722754fea client_f369cb89fc              72.416              0.626                  1.777 model_decline_risk|ctr_gap_vs_position_tier|model_and_rule_agree refresh_priority           140079           7.6                      20            down
    2 content_39881853ef0c client_f369cb89fc              71.945              0.628                  1.745 model_decline_risk|ctr_gap_vs_position_tier|model_and_rule_agree refresh_priority           112434           7.2                      20            down
    3 content_c84a0ab98e90 client_f369cb89fc              69.335             

## 2. Intended use and limits

**Intended use:** a weekly-refresh triage aid for a content strategist working down a limited
review queue, across this specific portfolio (32 clients, 30,000 pages, 90-day search snapshot).

**Where it stops being valid:**
- **Coverage is uneven.** 84% of `refresh_priority` flags concentrate in just 5 of 32 clients
  (see below) — before trusting that as "these clients need the most help," rule out a data or
  tracking artifact specific to them.
- **A quarter of the portfolio has thin data.** 26.6% of pages have under 100 impressions/90d;
  their scores are statistically unreliable regardless of confidence tier.
- **It does not generalize to a client it wasn't fit on** — this is the exact finding from ML-09;
  a newly onboarded client needs its own held-out check before this queue is trusted for it.
- **It is a decision-support tool, not a publishing tool.** It ranks review candidates; it does
  not know about seasonality, SERP feature changes, or factual accuracy.


In [2]:
high_thresh = df["final_action_score"].quantile(0.90)
med_thresh = df["final_action_score"].quantile(0.70)

def confidence(row):
    if row["final_action_score"] >= high_thresh and row["impressions_90d"] >= 500:
        return "high"
    if row["final_action_score"] >= med_thresh:
        return "medium"
    return "low"

df["confidence"] = df.apply(confidence, axis=1)
print("Confidence tier counts:")
print(df["confidence"].value_counts())
print()

# Coverage: which clients get the most "refresh_priority" flags -- is this queue balanced
# across the portfolio, or dominated by a few clients (a limit worth naming up front)?
top_clients = df[df["suggested_action"] == "refresh_priority"]["client_id"].value_counts().head(5)
print("Clients with the most refresh_priority flags (top 5 of 32):")
print(top_clients)
print(f"Those 5 clients hold {top_clients.sum()} of {(df['suggested_action']=='refresh_priority').sum()} total refresh_priority flags "
      f"({100*top_clients.sum()/(df['suggested_action']=='refresh_priority').sum():.0f}%).")
print()

low_data = (df["impressions_90d"] < 100).sum()
print(f"Pages with under 100 impressions/90d (thin data, score is unreliable regardless of tier): {low_data:,} ({100*low_data/len(df):.1f}%)")


Confidence tier counts:
confidence
low       21000
medium     6176
high       2824
Name: count, dtype: int64

Clients with the most refresh_priority flags (top 5 of 32):
client_id
client_6208ef0f77    466
client_19581e27de    461
client_3fdba35f04    216
client_7f2253d7e2    136
client_f74efabef1    123
Name: count, dtype: int64
Those 5 clients hold 1402 of 1721 total refresh_priority flags (81%).

Pages with under 100 impressions/90d (thin data, score is unreliable regardless of tier): 7,994 (26.6%)

## 3. Human review + the no-go list

**Never auto-action, route to mandatory human review instead:** pages updated in the last 14
days (re-flagging right after a refresh is noise, 3,428 pages), and pages under 20
impressions/90d (no reliable signal at all, 4,818 pages) — 23.6% of the portfolio combined,
now labeled `insufficient_data` rather than ranked. (No pages were excluded for being
"too new" — the starter dataset was already filtered to content 90+ days old before it ever
reached us.)

**What a person must always check before acting on a `refresh_priority` flag:** recent SERP
feature changes (featured snippets, PAA boxes eating clicks) that no column here captures;
seasonality; whether the content is still factually correct. **What must never be automated:**
generating or publishing rewritten content from this queue directly — it flags *what* to look at,
never *what to write*.


In [3]:
# No-go filters: never auto-action these, flag for mandatory human review instead.
too_new = df["content_age_days"] < 90          # not enough history to judge a "decline"
just_touched = df["days_since_last_update"] < 14   # already refreshed -- re-flagging is noise
too_thin = df["impressions_90d"] < 20          # essentially no signal at all

no_go_mask = too_new | just_touched | too_thin
print(f"No-go (excluded from actionable ranking, routed to insufficient_data instead):")
print(f"  too_new (age < 90d):              {too_new.sum():,}")
print(f"  just_touched (updated < 14d ago):  {just_touched.sum():,}")
print(f"  too_thin (impressions < 20/90d):   {too_thin.sum():,}")
print(f"  union (no double counting):        {no_go_mask.sum():,} of {len(df):,} ({100*no_go_mask.sum()/len(df):.1f}%)")

df.loc[no_go_mask, "suggested_action"] = "insufficient_data"
df.loc[no_go_mask, "confidence"] = "low"
print()
print("Updated action mix after applying the no-go filters:")
print(df["suggested_action"].value_counts())
print()
print("Human-review requirement (never auto-published from this queue alone):")
print("- Any 'refresh_priority' page still needs a human check for: recent rich-result/SERP")
print("  feature changes, seasonality, and whether content is factually current -- none of")
print("  those are in this dataset, so the model cannot see them.")
print("- Never auto-generate or auto-publish rewritten content from this queue; it flags")
print("  WHAT to review, not what to write.")


No-go (excluded from actionable ranking, routed to insufficient_data instead):
  too_new (age < 90d):              0
  just_touched (updated < 14d ago):  3,428
  too_thin (impressions < 20/90d):   4,818
  union (no double counting):        7,079 of 30,000 (23.6%)

Updated action mix after applying the no-go filters:
suggested_action
monitor              11665
refresh_review        9604
insufficient_data     7079
refresh_priority      1652
Name: count, dtype: int64

Human-review requirement (never auto-published from this queue alone):
- Any 'refresh_priority' page still needs a human check for: recent rich-result/SERP
  feature changes, seasonality, and whether content is factually current -- none of
  those are in this dataset, so the model cannot see them.
- Never auto-generate or auto-publish rewritten content from this queue; it flags
  WHAT to review, not what to write.

## 4. Monitoring / retrain triggers

Four triggers, each tied to a number this project already produced (not a guess):

1. **Performance drift** — recompute precision@50 monthly on a fresh held-out client split;
   alert if it drops materially below ML-09's audited, leakage-clean number (0.46) — that's
   the honest number to defend, not Week-5's inflated 0.74.
2. **Label base-rate drift** — current decline rate is 0.542; a swing past roughly +/-0.10 means
   the confidence thresholds (0.60, and the 90th/70th percentile cuts) need re-tuning.
3. **Portfolio concentration** — currently 84% of `refresh_priority` flags sit in 5 of 32
   clients; if a newly onboarded client instantly dominates the same way, check for a tracking
   artifact before trusting it.
4. **New-client generalization** — per ML-09, this model has no proven generalization to a
   client outside its training set; score new clients but hold their rows at "low" confidence
   until they've been through one honest held-out evaluation round.


In [4]:
# Monitoring / retrain triggers -- what would tell us this queue has gone stale.
print("Trigger 1 -- performance drift vs. the ML-09 audited baseline:")
print("  Recompute precision@50 on a fresh held-out client split monthly. Alert if it falls")
print("  more than 10 points below the ML-09 leakage-clean number (0.46) -- that's the number")
print("  to defend, not Week-5's inflated 0.74.")
print()

print("Trigger 2 -- label base-rate drift:")
print(f"  Current is_declining_label rate: {y.mean():.3f}. Recompute monthly; a shift past")
print("  roughly +/-0.10 means the portfolio's mix has changed enough that thresholds")
print("  (0.60 for model_decline_risk, the 90th/70th percentile confidence cuts) need review.")
print()

print("Trigger 3 -- portfolio concentration:")
current_priority = (df["suggested_action"] == "refresh_priority")
top_clients_now = df.loc[current_priority, "client_id"].value_counts().head(5)
share = 100 * top_clients_now.sum() / current_priority.sum()
print(f"  Currently {top_clients_now.sum()} of {current_priority.sum()} refresh_priority flags")
print(f"  ({share:.0f}%) sit in the top 5 of 32 clients. If a new client onboards and instantly")
print("  dominates the queue the same way, check for a data artifact (e.g. a tracking bug)")
print("  before trusting it.")
print()

print("Trigger 4 -- new clients need their own held-out check:")
print(f"  This model was fit on all {df['client_id'].nunique()} current clients. A newly onboarded")
print("  client has no guarantee the model generalizes to it (Week-9's whole point) -- score new")
print("  clients but flag their rows 'low' confidence until at least one honest evaluation round")
print("  has run with them included in a held-out test fold.")


Trigger 1 -- performance drift vs. the ML-09 audited baseline:
  Recompute precision@50 on a fresh held-out client split monthly. Alert if it falls
  more than 10 points below the ML-09 leakage-clean number (0.46) -- that's the number
  to defend, not Week-5's inflated 0.74.

Trigger 2 -- label base-rate drift:
  Current is_declining_label rate: 0.542. Recompute monthly; a shift past
  roughly +/-0.10 means the portfolio's mix has changed enough that thresholds
  (0.60 for model_decline_risk, the 90th/70th percentile confidence cuts) need review.

Trigger 3 -- portfolio concentration:
  Currently 1390 of 1652 refresh_priority flags
  (84%) sit in the top 5 of 32 clients. If a new client onboards and instantly
  dominates the queue the same way, check for a data artifact (e.g. a tracking bug)
  before trusting it.

Trigger 4 -- new clients need their own held-out check:
  This model was fit on all 32 current clients. A newly onboarded
  client has no guarantee the model generalizes to i

## 5. Exports for the paper

Wrote `work/outputs/w07_action_queue_summary.json` (the action/confidence mix, no-go counts,
and — importantly — a field that points back at ML-09's audited precision@50 rather than
Week-5's inflated number, so the paper cites the honest figure by construction) and
`work/outputs/w07_action_mix.svg` (a simple bar chart of the post-no-go-filter action mix) for
reuse in the capstone paper.


In [5]:
import json

# JSON receipt: the numbers this week's paper section will cite.
summary = {
    "notebook": "w07_action_playbook.ipynb",
    "total_pages": int(len(df)),
    "clients": int(df["client_id"].nunique()),
    "action_mix": df["suggested_action"].value_counts().to_dict(),
    "confidence_mix": df["confidence"].value_counts().to_dict(),
    "no_go_excluded": int(no_go_mask.sum()),
    "thin_data_pct": round(100 * (df["impressions_90d"] < 100).mean(), 1),
    "top5_client_share_of_refresh_priority_pct": round(share, 1),
    "expected_precision_at_50_source": "ML-09 leakage-clean held-out audit (0.46), not Week-5's inflated in-sample-adjacent 0.74",
    "model": "logistic_regression, leakage-clean feature set (no impressions_90d/clicks_90d/sessions_90d), refit on all 32 clients for production scoring",
}
with open("work/outputs/w07_action_queue_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Wrote work/outputs/w07_action_queue_summary.json")
print(json.dumps(summary, indent=2))

# One chart for the paper: the action mix, as a minimal inline SVG bar chart (no plotting
# library needed, and it's a static asset so it's safe to commit).
def svg_bar_chart(title, labels, values, path):
    width, height = 720, 320
    margin_left, margin_top, margin_bottom = 190, 60, 40
    plot_w, plot_h = width - margin_left - 40, height - margin_top - margin_bottom
    max_v = max(values) if values else 1
    bar_h = plot_h / max(len(values), 1) - 12
    lines = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
             '<rect width="100%" height="100%" fill="#ffffff"/>',
             f'<text x="{width/2}" y="30" text-anchor="middle" font-family="Arial" font-size="18" fill="#16232a">{title}</text>']
    for i, (label, value) in enumerate(zip(labels, values)):
        y = margin_top + i * (bar_h + 12)
        w = (value / max_v) * plot_w
        lines.append(f'<text x="{margin_left-10}" y="{y+bar_h*0.65:.1f}" text-anchor="end" font-family="Arial" font-size="13" fill="#27343b">{label}</text>')
        lines.append(f'<rect x="{margin_left}" y="{y:.1f}" width="{w:.1f}" height="{bar_h:.1f}" fill="#426B69" rx="3"/>')
        lines.append(f'<text x="{margin_left+w+8:.1f}" y="{y+bar_h*0.65:.1f}" font-family="Arial" font-size="13" fill="#27343b">{value:,}</text>')
    lines.append("</svg>")
    with open(path, "w") as f:
        f.write("\n".join(lines))

mix = df["suggested_action"].value_counts()
svg_bar_chart("Action mix (after no-go filters)", mix.index.tolist(), mix.values.tolist(),
              "work/outputs/w07_action_mix.svg")
print("Wrote work/outputs/w07_action_mix.svg")


Wrote work/outputs/w07_action_queue_summary.json
{
  "notebook": "w07_action_playbook.ipynb",
  "total_pages": 30000,
  "clients": 32,
  "action_mix": {
    "monitor": 11665,
    "refresh_review": 9604,
    "insufficient_data": 7079,
    "refresh_priority": 1652
  },
  "confidence_mix": {
    "low": 22082,
    "medium": 5248,
    "high": 2670
  },
  "no_go_excluded": 7079,
  "thin_data_pct": 26.6,
  "top5_client_share_of_refresh_priority_pct": 84.1,
  "expected_precision_at_50_source": "ML-09 leakage-clean held-out audit (0.46), not Week-5's inflated in-sample-adjacent 0.74",
  "model": "logistic_regression, leakage-clean feature set (no impressions_90d/clicks_90d/sessions_90d), refit on all 32 clients for production scoring"
}
Wrote work/outputs/w07_action_mix.svg

## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
